# Assignment 3 — Dimensionality Reduction with PCA
**Dataset:** UCI Student Performance (`student-por.csv`)

**Aim:** Encode + standardize student features (excluding G1/G2/G3), apply PCA, quantify explained variance, find components for 95% variance, visualize PC1 vs PC2, and interpret loadings.


## 1. Library imports

In [1]:
import io, os
import urllib.request, zipfile
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
os.makedirs("outputs", exist_ok=True)
print("random_state =", RANDOM_STATE)

random_state = 42


## 2. Dataset loading (official UCI source)

In [2]:
UCI_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip"
raw = urllib.request.urlopen(UCI_URL, timeout=60).read()
zf = zipfile.ZipFile(io.BytesIO(raw))
with zf.open("student-por.csv") as f:
    df = pd.read_csv(f, sep=";")
print("Shape:", df.shape)

Shape: (649, 33)


## 3. Dataset preview

In [3]:
df.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13


In [4]:
print("Missing total:", int(df.isnull().sum().sum()))
print(df.dtypes.value_counts())

Missing total: 0
str      17
int64    16
Name: count, dtype: int64


## 4. Data preprocessing — encoded + standardized features (exclude G1/G2/G3)

In [5]:
LEAK = ["G1", "G2", "G3"]
X_raw = df.drop(columns=LEAK)
cat_cols = X_raw.select_dtypes(include=["object"]).columns.tolist()
num_cols = X_raw.select_dtypes(exclude=["object"]).columns.tolist()
print(f"Features: {X_raw.shape[1]} (excluded {LEAK}); numeric={len(num_cols)}, categorical={len(cat_cols)}")
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)])
X = preprocess.fit_transform(X_raw)
feat_names = list(preprocess.get_feature_names_out())
print("Processed matrix:", X.shape)
print("Example features:", feat_names[:8])

Features: 30 (excluded ['G1', 'G2', 'G3']); numeric=13, categorical=17
Processed matrix: (649, 56)
Example features: ['num__age', 'num__Medu', 'num__Fedu', 'num__traveltime', 'num__studytime', 'num__failures', 'num__famrel', 'num__freetime']


C:\Users\ns479\AppData\Local\Temp\ipykernel_15512\4142146689.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_raw.select_dtypes(include=["object"]).columns.tolist()


## 5. PCA model implementation

In [6]:
pca = PCA(random_state=RANDOM_STATE)
Z = pca.fit_transform(X)
evr = pca.explained_variance_ratio_
print(f"Components: {pca.n_components_}; total variance (should be 1.0): {evr.sum():.6f}")
var_table = pd.DataFrame({
    "component": [f"PC{i+1}" for i in range(len(evr))],
    "explained_variance_ratio": evr,
    "cumulative_variance": np.cumsum(evr),
    "eigenvalue": pca.explained_variance_})
print(var_table.head(15).round(4).to_string(index=False))
var_table.to_csv("outputs/explained_variance.csv", index=False)
n95 = int(np.argmax(np.cumsum(evr) >= 0.95) + 1)
print(f"\nComponents required for >=95% variance: {n95} (cumulative={np.cumsum(evr)[n95-1]:.4f})")

Components: 56; total variance (should be 1.0): 1.000000
component  explained_variance_ratio  cumulative_variance  eigenvalue
      PC1                    0.1198               0.1198      2.4077
      PC2                    0.1046               0.2244      2.1034
      PC3                    0.0685               0.2929      1.3761
      PC4                    0.0605               0.3534      1.2160
      PC5                    0.0551               0.4085      1.1073
      PC6                    0.0475               0.4559      0.9544
      PC7                    0.0472               0.5032      0.9493
      PC8                    0.0442               0.5474      0.8892
      PC9                    0.0414               0.5888      0.8322
     PC10                    0.0351               0.6239      0.7055
     PC11                    0.0307               0.6546      0.6166
     PC12                    0.0277               0.6823      0.5572
     PC13                    0.0248           

## 6. Evaluation — scree plot + cumulative variance graph

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(range(1, len(evr)+1), evr)
axes[0].set_xlabel("Principal component"); axes[0].set_ylabel("Explained variance ratio")
axes[0].set_title("Scree plot")
axes[1].plot(range(1, len(evr)+1), np.cumsum(evr), marker="o")
axes[1].axhline(0.95, color="red", linestyle="--", label="95% threshold")
axes[1].axvline(n95, color="red", linestyle=":", label=f"n95 = {n95}")
axes[1].set_xlabel("Number of components"); axes[1].set_ylabel("Cumulative variance")
axes[1].set_title("Cumulative explained variance"); axes[1].legend()
plt.suptitle("PCA explained variance")
plt.tight_layout(); plt.savefig("outputs/scree_cumulative.png", dpi=150); plt.close()
print("Saved: outputs/scree_cumulative.png")

Saved: outputs/scree_cumulative.png


## 7. Tables and graphs — PC1 vs PC2 visualization (coloured by Pass/Fail)

In [8]:
df["Pass"] = (df["G3"] >= 10).astype(int)
plt.figure(figsize=(8, 6))
for v, lab in [(0, "Fail (G3<10)"), (1, "Pass (G3>=10)")]:
    m = df["Pass"].to_numpy() == v
    plt.scatter(Z[m, 0], Z[m, 1], s=20, alpha=0.65, label=lab)
plt.xlabel(f"PC1 ({evr[0]:.1%} variance)")
plt.ylabel(f"PC2 ({evr[1]:.1%} variance)")
plt.title("PC1 vs PC2 (colour = early-prediction target Pass/Fail)")
plt.legend(); plt.tight_layout()
plt.savefig("outputs/pc1_pc2.png", dpi=150); plt.close()
print("Saved: outputs/pc1_pc2.png")
print(f"PC1+PC2 capture {evr[0]+evr[1]:.2%} of variance.")

Saved: outputs/pc1_pc2.png
PC1+PC2 capture 22.44% of variance.


## 8. PCA loading interpretation (PC1 / PC2 top contributors)

In [9]:
loadings = pd.DataFrame(pca.components_[:5].T, index=feat_names,
                                columns=[f"PC{i+1}" for i in range(5)])
print("Top +5 / -5 loadings for PC1:")
print(loadings["PC1"].sort_values(ascending=False).head(5).round(4))
print(loadings["PC1"].sort_values(ascending=True).head(5).round(4))
print("\nTop +5 / -5 loadings for PC2:")
print(loadings["PC2"].sort_values(ascending=False).head(5).round(4))
print(loadings["PC2"].sort_values(ascending=True).head(5).round(4))
loadings.to_csv("outputs/pca_loadings_pc1_pc5.csv")
# Bar chart of top-10 absolute loadings for PC1 and PC2
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, pc in zip(axes, ["PC1", "PC2"]):
    top = loadings[pc].abs().sort_values(ascending=False).head(10).index
    vals = loadings.loc[top, pc]
    ax.barh(top[::-1], vals.loc[top[::-1]])
    ax.set_title(f"Top-10 loadings: {pc}")
plt.tight_layout(); plt.savefig("outputs/pca_loadings.png", dpi=150); plt.close()
print("Saved: outputs/pca_loadings.png, outputs/pca_loadings_pc1_pc5.csv")

Top +5 / -5 loadings for PC1:
num__Walc        0.3874
num__Dalc        0.3778
num__failures    0.3103
num__goout       0.2760
num__age         0.2606
Name: PC1, dtype: float64
num__Medu         -0.3302
num__Fedu         -0.2897
num__studytime    -0.2563
cat__school_GP    -0.0902
cat__higher_yes   -0.0765
Name: PC1, dtype: float64

Top +5 / -5 loadings for PC2:
num__Fedu     0.4524
num__Medu     0.4511
num__Walc     0.3743
num__Dalc     0.3209
num__goout    0.2906
Name: PC2, dtype: float64
num__traveltime    -0.2236
cat__sex_F         -0.1234
cat__school_MS     -0.1011
cat__internet_no   -0.1001
num__failures      -0.0899
Name: PC2, dtype: float64


Saved: outputs/pca_loadings.png, outputs/pca_loadings_pc1_pc5.csv


## 9. Result and conclusion
- Encoded/standardized features via `ColumnTransformer`; full PCA fitted with `random_state=42`.
- Explained-variance table (`outputs/explained_variance.csv`), scree + cumulative plot, PC1-vs-PC2 scatter and loading charts saved to `outputs/`.
- Number of components for 95% variance (`n95`) is computed at runtime and marked on the cumulative graph.
